In [1]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DoubleType
from pyspark.sql.functions import * 
from pyspark.sql import Row
from datetime import datetime


StatementMeta(, 885b1189-bf83-4b25-8731-7a6f25e5d884, 3, Finished, Available, Finished)

**Import tables from silver lakehouse**

In [2]:
agents_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.agents")
applicants_addresses_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.applicants_addresses")
applicants_banking_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.applicants_banking")
applicants_contacts_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.applicants_contacts")
applicants_employment_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.applicants_employment")
applicants_health_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.applicants_health")
claim_processing_details_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.claim_processng_details")
claims_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.claims")
demographic_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.demographic")
fact_applicants_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.fact_applicants")
insurance_policies_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.insurance_policies")
nationality_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.nationality")
payment_history_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.payment_history")
policies_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.policies")
policies_dates_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.policies_dates")
policy_coverages_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.policy_coverages")
reinsurance_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.reinsurance")
reinsurance_companies_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.reinsurance_companies")

StatementMeta(, 885b1189-bf83-4b25-8731-7a6f25e5d884, 4, Finished, Available, Finished)

**Put all tables into one list and under their prospective schemas**

In [3]:
# Organize tables into categories
tables = {
    "claims": [
        "claims",
        "claim_processing_details"
    ],
    "applicants": [
        "applicants_addresses",
        "applicants_banking",
        "applicants_contacts",
        "applicants_employment",
        "applicants_health",
        "demographic",
        "fact_applicants",
        "nationality"
    ],
    "premiums": [
        "payment_history"
    ],
    "policies": [
        "agents",
        "insurance_policies",
        "policies",
        "policies_dates",
        "policy_coverages",
        "reinsurance",
        "reinsurance_companies"
    ]
}

all_tables_flat = []
for category, table_list in tables.items():
    all_tables_flat.extend(table_list)

print("Tables organized by category:")
print("=" * 50)
for category, table_list in tables.items():
    print(f"\n{category.upper()} ({len(table_list)} tables):")
    for table in table_list:
        print(f"  • {table}")

print(f"\nTotal tables across all categories: {len(all_tables_flat)}")

StatementMeta(, 885b1189-bf83-4b25-8731-7a6f25e5d884, 5, Finished, Available, Finished)

Tables organized by category:

CLAIMS (2 tables):
  • claims
  • claim_processing_details

APPLICANTS (8 tables):
  • applicants_addresses
  • applicants_banking
  • applicants_contacts
  • applicants_employment
  • applicants_health
  • demographic
  • fact_applicants
  • nationality

PREMIUMS (1 tables):
  • payment_history

POLICIES (7 tables):
  • agents
  • insurance_policies
  • policies
  • policies_dates
  • policy_coverages
  • reinsurance
  • reinsurance_companies

Total tables across all categories: 18


In [4]:
table_dataframes = {
    "agents": agents_df,
    "applicants_addresses": applicants_addresses_df,
    "applicants_banking": applicants_banking_df,
    "applicants_contacts": applicants_contacts_df,
    "applicants_employment": applicants_employment_df,
    "applicants_health": applicants_health_df,
    "claim_processing_details": claim_processing_details_df,
    "claims": claims_df,
    "demographic": demographic_df,
    "fact_applicants": fact_applicants_df,
    "insurance_policies": insurance_policies_df,
    "nationality": nationality_df,
    "payment_history": payment_history_df,
    "policies": policies_df,
    "policies_dates": policies_dates_df,
    "policy_coverages": policy_coverages_df,
    "reinsurance": reinsurance_df,
    "reinsurance_companies": reinsurance_companies_df
}

StatementMeta(, 885b1189-bf83-4b25-8731-7a6f25e5d884, 6, Finished, Available, Finished)

In [5]:
# Define schema for entry validation DataFrame
entry_schema = StructType([
    StructField("table_name", StringType(), True),
    StructField("no_of_duplicates", IntegerType(), True),
    StructField("no_of_columns", IntegerType(), True),
    StructField("no_of_rows", IntegerType(), True),
    StructField("validation_timestamp", TimestampType(), True),
    StructField("null_percentage", DoubleType(), True),
    StructField("unique_key_check", StringType(), True),
    StructField("data_type_validation", StringType(), True)
])

# Define schema for exit validation DataFrame
exit_schema = StructType([
    StructField("table_name", StringType(), True),
    StructField("no_of_duplicates", IntegerType(), True),
    StructField("no_of_columns", IntegerType(), True),
    StructField("no_of_rows", IntegerType(), True),
    StructField("validation_timestamp", TimestampType(), True),
    StructField("null_percentage", DoubleType(), True),
    StructField("unique_key_check", StringType(), True),
    StructField("data_type_validation", StringType(), True),
    StructField("processing_status", StringType(), True)
])

# Create empty DataFrames with explicit schemas
entry_validation_df = spark.createDataFrame([], schema=entry_schema)
exit_validation_df = spark.createDataFrame([], schema=exit_schema)

# Display schema to verify
print("Entry Validation DataFrame Schema:")
entry_validation_df.printSchema()
print("\nExit Validation DataFrame Schema:")
exit_validation_df.printSchema()

StatementMeta(, 885b1189-bf83-4b25-8731-7a6f25e5d884, 7, Finished, Available, Finished)

Entry Validation DataFrame Schema:
root
 |-- table_name: string (nullable = true)
 |-- no_of_duplicates: integer (nullable = true)
 |-- no_of_columns: integer (nullable = true)
 |-- no_of_rows: integer (nullable = true)
 |-- validation_timestamp: timestamp (nullable = true)
 |-- null_percentage: double (nullable = true)
 |-- unique_key_check: string (nullable = true)
 |-- data_type_validation: string (nullable = true)


Exit Validation DataFrame Schema:
root
 |-- table_name: string (nullable = true)
 |-- no_of_duplicates: integer (nullable = true)
 |-- no_of_columns: integer (nullable = true)
 |-- no_of_rows: integer (nullable = true)
 |-- validation_timestamp: timestamp (nullable = true)
 |-- null_percentage: double (nullable = true)
 |-- unique_key_check: string (nullable = true)
 |-- data_type_validation: string (nullable = true)
 |-- processing_status: string (nullable = true)



In [7]:
# Corrected validate_table function for ENTRY validation (without processing_status)
def validate_table(df, table_name, validation_type="entry"):
    """
    Validate a DataFrame and return a Row for the validation DataFrame
    """
    from pyspark.sql.functions import col
    
    # Calculate metrics
    no_of_rows = df.count()
    no_of_columns = len(df.columns)
    no_of_duplicates = df.count() - df.distinct().count() if df.count() > 0 else 0
    
    # Calculate null percentage (using first column as example)
    first_col = df.columns[0] if df.columns else None
    null_percentage = None
    if first_col:
        null_count = df.filter(col(first_col).isNull()).count()
        null_percentage = float(null_count / no_of_rows * 100) if no_of_rows > 0 else 0.0
    
    # For ENTRY validation - create Row with 8 fields (matching entry_schema)
    if validation_type == "entry":
        return Row(
            table_name=table_name,
            no_of_duplicates=no_of_duplicates,
            no_of_columns=no_of_columns,
            no_of_rows=no_of_rows,
            validation_timestamp=datetime.now(),
            null_percentage=null_percentage,
            unique_key_check="PASS" if no_of_duplicates == 0 else "FAIL",
            data_type_validation="PASS"
        )
    else:
        return Row(
            table_name=table_name,
            no_of_duplicates=no_of_duplicates,
            no_of_columns=no_of_columns,
            no_of_rows=no_of_rows,
            validation_timestamp=datetime.now(),
            null_percentage=null_percentage,
            unique_key_check="PASS" if no_of_duplicates == 0 else "FAIL",
            data_type_validation="PASS",
            processing_status="COMPLETED"
        )

# Validate each table and collect validation records for 
validation_records = []

for table_name, df in table_dataframes.items():
    print(f"Validating table: {table_name}")
    validation_record = validate_table(df, table_name, "entry")  
    validation_records.append(validation_record)

# Create a DataFrame from the collected records using ENTRY schema
if validation_records:
    entry_validation_df = spark.createDataFrame(validation_records, schema=entry_schema)
    print("Entry Validation Results:")
    entry_validation_df.show(truncate=False)
else:
    print("No validation records created.")

StatementMeta(, 885b1189-bf83-4b25-8731-7a6f25e5d884, 9, Finished, Available, Finished)

Validating table: agents
Validating table: applicants_addresses
Validating table: applicants_banking
Validating table: applicants_contacts
Validating table: applicants_employment
Validating table: applicants_health
Validating table: claim_processing_details
Validating table: claims
Validating table: demographic
Validating table: fact_applicants
Validating table: insurance_policies
Validating table: nationality
Validating table: payment_history
Validating table: policies
Validating table: policies_dates
Validating table: policy_coverages
Validating table: reinsurance
Validating table: reinsurance_companies
Entry Validation Results:
+------------------------+----------------+-------------+----------+--------------------------+---------------+----------------+--------------------+
|table_name              |no_of_duplicates|no_of_columns|no_of_rows|validation_timestamp      |null_percentage|unique_key_check|data_type_validation|
+------------------------+----------------+-------------+----